# 00 · Setup & environment check
Run this first in every new Colab session. It mounts Drive, loads the project code, installs the light requirements and verifies the GPU.

**Before you run:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

> Free Colab: sessions are temporary and the disk is wiped. Everything important is written to Google Drive, and every long job in this project is resumable.

In [ ]:
# --- Bootstrap: mount Drive, get the code, make `heritage` importable ---
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/artefact-restoration-3d.git"  # <- set after you push to GitHub
REPO_DIR = Path("/content/artefact-restoration-3d")
DRIVE_CODE = Path("/content/drive/MyDrive/artefact-restoration-3d/code")   # fallback: upload the repo folder here

from google.colab import drive
drive.mount("/content/drive")

if "YOUR_USERNAME" not in REPO_URL:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif DRIVE_CODE.exists():
    # Re-sync from Drive on EVERY run so an updated code folder is never ignored
    shutil.copytree(DRIVE_CODE, REPO_DIR, dirs_exist_ok=True)
else:
    raise SystemExit("Set REPO_URL (after pushing to GitHub) or upload the repo folder to " + str(DRIVE_CODE))

sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
print("Repo ready at", REPO_DIR)


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from heritage import colab, paths
cfg = paths.load_config()
P = paths.get_paths(cfg).ensure()
report = colab.check_env()
colab.seed_everything(cfg['project']['seed'])
print('\nDrive workspace :', P.drive)
print('Local scratch   :', P.local)

### If the check above shows a problem
| Symptom | Fix |
|---|---|
| `No GPU` | Runtime ▸ Change runtime type ▸ T4. If none is offered, wait a while or switch to the Kaggle notebook fallback. |
| `numpy<->torch conversion FAILED` | `!pip install 'numpy<2'` then **Runtime ▸ Restart session** and re-run (same bug we hit on the DGX). |
| Drive not mounting | Re-run the first cell and approve the pop-up. |

In [ ]:
import json
(P.reports).mkdir(parents=True, exist_ok=True)
(P.reports / 'runtime_check.json').write_text(json.dumps(report, indent=2, default=str))
print('Saved', P.reports / 'runtime_check.json')